# B1.3 · Threat modelling from the architecture map

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *Security of AI*

Builds on **[B1.2 · Component summarisation and architecture synthesis](https://spbreed.github.io/cyber-commons/lessons/B1.2.html)**.

| | |
|---|---|
| Open-source tooling | OWASP Threat Dragon |
| Open-weight models | GLM-4.6 |
| Frontier models | Claude Sonnet 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A threat model produced in a workshop describes the system as it was on the day of the workshop. By the second sprint it describes something that no longer exists, and nobody notices, because nothing re-reads it.

> **At CyberTravels.** The threat model that said “TripBot answers questions” is still on file. Deriving it from the architecture on every release is what would have caught the refund endpoint appearing.

## 2 · The framework

```
   architecture map            threat model
   +----------------+          +---------------------------+
   | components     |  derive  | entry points              |
   | flows          | -------> | trust boundaries crossed  |
   | trust levels   |          | assets reachable          |
   | sinks          |          | abuse cases               |
   +----------------+          +---------------------------+
           |                              |
        regenerated per release      DIFFED against the last one

   the diff is the product: "what did this pull request just introduce"
```

Phase 2 opens with the stage everyone claims to do and almost nobody re-runs.

**Stage 5 — Threat modelling.** Read the architecture map from stage 4 and
derive, mechanically: high-value assets, untrusted entry points, and the attack
vectors that connect them.

The word doing the work is *mechanically*. A threat model produced by hand in a
workshop is a snapshot; it is stale the moment an entry point is added, and
adding an entry point is a Tuesday. A threat model **derived from the map** is
regenerated whenever the map changes, so the useful artefact is not the model —
it is the **diff between two models**.

That reframing is what makes threat modelling a pipeline stage rather than a
document. It also means the output has to be data: ranked, machine-readable, and
consumable by stage 6, which allocates the analysis budget against it.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Stage 5 — derive threats from the map

In [ ]:
from dataclasses import dataclass, field
from collections import defaultdict

# The stage-4 output, as data.
ARCH = {
 "entry_points": [
   {"unit": "get_report",   "component": "src/web", "auth": "session"},
   {"unit": "upload_doc",   "component": "src/web", "auth": "session"},
   {"unit": "health",       "component": "src/web", "auth": "none"},
 ],
 "flows": [("get_report", "load_report"), ("get_report", "render"),
           ("upload_doc", "store"), ("load_report", "execute"),
           ("store", "open")],
 "sinks": [{"unit": "load_report", "sink": "execute", "resource": "database"},
           {"unit": "store", "sink": "open", "resource": "filesystem"}],
 "assets": {"database": {"data": ("customer", "financial"), "value": 5},
            "filesystem": {"data": ("documents",), "value": 3},
            "session_store": {"data": ("credentials",), "value": 5}},
}

VECTOR_FOR = {
 "database":   [("CWE-89",  "SQL injection", 5)],
 "filesystem": [("CWE-22",  "path traversal", 4), ("CWE-434", "unrestricted upload", 4)],
 "shell":      [("CWE-78",  "command injection", 5)],
}

def reachable(entry, flows):
    adj = defaultdict(list)
    for a, b in flows: adj[a].append(b)
    seen, stack = set(), [entry]
    while stack:
        n = stack.pop()
        for m in adj[n]:
            if m not in seen: seen.add(m); stack.append(m)
    return seen

def threat_model(arch):
    threats = []
    for ep in arch["entry_points"]:
        reach = reachable(ep["unit"], arch["flows"])
        for sink in arch["sinks"]:
            if sink["unit"] not in reach: continue
            asset = arch["assets"].get(sink["resource"], {"value": 1, "data": ()})
            for cwe, name, base in VECTOR_FOR.get(sink["resource"], []):
                score = base + asset["value"] + (2 if ep["auth"] == "none" else 0)
                threats.append({
                    "entry": ep["unit"], "auth": ep["auth"],
                    "sink": sink["unit"], "resource": sink["resource"],
                    "cwe": cwe, "vector": name, "score": score,
                    "path": f"{ep['unit']} → … → {sink['unit']}",
                    "data_at_risk": list(asset["data"])})
    # deterministic on every machine: score first, then a stable tiebreak
    return sorted(threats, key=lambda t: (-t["score"], t["cwe"], t["entry"], t["sink"]))

TM = threat_model(ARCH)
print(f"{'entry':13s}{'sink':13s}{'cwe':9s}{'vector':22s}{'score':>6}  data at risk")
print("-" * 88)
for t in TM:
    print(f"{t['entry']:13s}{t['sink']:13s}{t['cwe']:9s}{t['vector']:22s}"
          f"{t['score']:>6}  {t['data_at_risk']}")
print(f"\n{len(TM)} threats derived. No human wrote this; it fell out of the map.")

## 4 · Where it breaks — the model that was true last quarter

Add one entry point. The hand-written threat model does not change, because documents do not change themselves.

In [ ]:
ARCH_V2 = {**ARCH,
 "entry_points": ARCH["entry_points"] + [
   {"unit": "admin_export", "component": "src/web", "auth": "none"}],
 "flows": ARCH["flows"] + [("admin_export", "load_report"),
                           ("admin_export", "store")]}

TM2 = threat_model(ARCH_V2)

def diff(before, after):
    key = lambda t: (t["entry"], t["sink"], t["cwe"])
    b = {key(t): t for t in before}
    a = {key(t): t for t in after}
    # sorted() over a set difference is NOT deterministic across processes:
    # set iteration order depends on PYTHONHASHSEED, and a stable sort then
    # preserves that order for equal scores. Sort the keys first.
    return {"new": [a[k] for k in sorted(a.keys() - b.keys())],
            "removed": [b[k] for k in sorted(b.keys() - a.keys())],
            "max_before": max(t["score"] for t in before),
            "max_after": max(t["score"] for t in after)}

d = diff(TM, TM2)
print(f"threats before {len(TM)} → after {len(TM2)}")
print(f"max severity   {d['max_before']} → {d['max_after']}")
print("\nNEW THREATS:")
# full tiebreak, so equal scores order the same way on every machine
for t in sorted(d["new"], key=lambda t: (-t["score"], t["cwe"], t["entry"], t["sink"])):
    print(f"   [{t['score']:>2}] {t['cwe']:9s}{t['path']:34s}auth={t['auth']}")
print("\nOne unauthenticated handler introduced 3 new threats, two of them")
print("higher-scoring than anything in the original model.")
assert d["new"] and d["max_after"] >= d["max_before"]

## 5 · The control — regenerate on every map change, and gate on the delta

In [ ]:
def threat_gate(before, after, max_new_critical=0, critical_at=11):
    d = diff(before, after)
    new_crit = [t for t in d["new"] if t["score"] >= critical_at]
    ok = len(new_crit) <= max_new_critical
    return ok, {"new_threats": len(d["new"]), "new_critical": len(new_crit),
                "detail": [f"{t['cwe']} via {t['path']} (score {t['score']})"
                           for t in new_crit]}

ok, info = threat_gate(TM, TM2)
print(f"CI gate: {'PASS' if ok else 'FAIL'}")
for k, v in info.items(): print(f"   {k:14s}{v}")

print("\nafter requiring auth on the new handler:")
ARCH_V3 = {**ARCH_V2,
 "entry_points": [{**e, "auth": "session"} if e["unit"] == "admin_export" else e
                  for e in ARCH_V2["entry_points"]]}
TM3 = threat_model(ARCH_V3)
ok3, info3 = threat_gate(TM, TM3)
print(f"CI gate: {'PASS' if ok3 else 'FAIL'}   new_critical={info3['new_critical']}")
print("\nThe gate did not ask anyone to write a document. It compared two")
print("generated models and refused a specific, named regression.")

## What you just proved

Six threats are derived from the map, ranked by combined vector, asset value and authentication, with SQL injection against customer and financial data scoring highest. Adding one unauthenticated handler produces three new threats and raises the maximum score. The CI gate fails on the new criticals and passes once the handler requires a session.

## Your turn

Wire the threat diff into CI for one service: regenerate on every merge and fail when a new critical path appears. It is the cheapest form of continuous threat modelling that exists, and it needs no workshop.

---

**Next → [B1.4 · Strategic planning and agent allocation](https://spbreed.github.io/cyber-commons/lessons/B1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*